# M1.S1 - Fundamentals of HPC
## From a serial problem to a supercomputer

The session starts from a simple question:

> **When does an ordinary computing problem become an HPC problem?**

HPC becomes relevant when a workload reaches a practical limit in **time, size, data or complexity**, and additional computing resources can help solve the problem effectively.

### What you will practice

By the end of the notebook you should be able to:

1. decide whether a workload is plausibly an HPC problem;
2. distinguish **time-to-solution** from **throughput**;
3. distinguish **concurrency**, **data parallelism** and **task parallelism**;
4. establish a trustworthy **serial baseline** before talking about speedup;
5. use **Amdahl's Law** to reason about scaling limits;
6. identify likely bottlenecks: compute, memory, network, storage or software;
7. inspect the SciTech environment safely without launching heavy computation.

Throughout the notebook use the same cycle:

> **PREDICT -> RUN -> OBSERVE -> EXPLAIN**

## 1 - When is a problem an HPC problem?

A useful first approximation is:

> **HPC problem = a computational limit + a scalable way of attacking it**

A slow program is **not automatically** an HPC problem. Sometimes the right answer is a better algorithm rather than more hardware.

### Predict

For each case, decide **YES / MAYBE / NO** and identify the first likely limit.

| Workload | Your answer | First likely limit |
|---|---|---|
| Sort 50,000 integers once | ? | ? |
| Run 10 million Monte Carlo scenarios | ? | ? |
| Simulate airflow around an aircraft at high resolution | ? | ? |
| Search patterns across 10 PB of telescope data | ? | ? |
| Train one large AI model on 1,024 GPUs | ? | ? |

<details>
<summary><strong>Show explanation</strong></summary>

- **Sort 50,000 integers once:** usually **NO**. A normal computer handles this easily.
- **10 million Monte Carlo scenarios:** often **YES/MAYBE**, depending on scenario cost and deadline. It exposes abundant independent work.
- **High-resolution aircraft CFD:** usually **YES**. Compute, memory and communication can all become limiting.
- **Search 10 PB:** **YES**. Data volume, storage bandwidth, memory hierarchy and distributed processing dominate.
- **Large AI model on 1,024 GPUs:** **YES**. Compute, accelerator memory and network communication all matter.

The important part is not the label. It is identifying **what practical limit is reached** and whether the work can be decomposed.

</details>

## 2 - Time-to-solution and throughput are different goals

The same cluster can be used for different performance objectives.

**Time-to-solution**
- finish one important job sooner;
- example: produce a weather forecast before its deadline.

**Throughput**
- finish more independent jobs per unit time;
- example: evaluate thousands of independent parameter combinations.

The following small experiment shows the difference.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def independent_job(job_id, delay=0.25):
    start = time.perf_counter()
    time.sleep(delay)
    return job_id, time.perf_counter() - start

N_JOBS = 8
DELAY = 0.25

print("Each toy job takes about", DELAY, "seconds.")
print("Number of independent jobs:", N_JOBS)

### Predict

If each individual task still needs about 0.25 s, what changes when we allow four tasks to make progress concurrently?

- Does the **latency of one individual task** become four times smaller?
- Does the **total batch completion time** become smaller?

In [ ]:
t0 = time.perf_counter()
serial_results = [independent_job(i, DELAY) for i in range(N_JOBS)]
serial_batch = time.perf_counter() - t0

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    concurrent_results = list(pool.map(lambda i: independent_job(i, DELAY), range(N_JOBS)))
concurrent_batch = time.perf_counter() - t0

avg_serial_task = sum(t for _, t in serial_results) / len(serial_results)
avg_concurrent_task = sum(t for _, t in concurrent_results) / len(concurrent_results)

print(f"Average individual task latency, serial batch:     {avg_serial_task:.3f} s")
print(f"Average individual task latency, concurrent batch: {avg_concurrent_task:.3f} s")
print(f"Total serial batch time:                            {serial_batch:.3f} s")
print(f"Total concurrent batch time:                        {concurrent_batch:.3f} s")
print(f"Throughput improvement for the batch:               {serial_batch/concurrent_batch:.2f}x")

### Observe and explain

The latency of one sleeping task remains approximately the same. What improved is **throughput**: several independent tasks overlap, so the whole batch finishes sooner.

This is also a useful reminder:

> **Concurrency is not automatically CPU parallelism.**

The toy tasks spend their time waiting. Later in the course we will use multiple CPU cores, multiple processes and GPUs for real simultaneous computation.

## 3 - Concurrency, data parallelism and task parallelism

### Concurrency vs parallelism

- **Concurrency:** several tasks make progress during the same time window. They can be interleaved.
- **Parallelism:** several operations execute at the same instant on different execution resources.

### Two common ways to expose parallel work

**Data parallelism**
- same operation;
- different pieces of data;
- example: update different cells of a weather grid.

**Task parallelism**
- different tasks or stages;
- execute at the same time;
- example: simulation, analysis and I/O overlap.

### Quick classification

Classify each example before opening the explanation.

1. Matrix multiplication, where different output tiles are computed simultaneously.
2. A parameter sweep with 5,000 independent parameter sets.
3. A workflow where one stage reads data, another computes, and another writes results.
4. Image filtering, where the same kernel is applied to different image regions.

<details>
<summary><strong>Show explanation</strong></summary>

1. Primarily **data parallelism**.
2. Usually **task/job parallelism** across independent simulations.
3. **Task/pipeline parallelism**.
4. Primarily **data parallelism**.

Many real applications combine more than one form.

</details>

## 4 - Rule #1 of performance work: establish a serial baseline

Before claiming a speedup, record what **one correct serial run** costs.

Use:

- the same input;
- the same correctness criterion;
- the same measurement method;
- repeated measurements when timing is noisy;
- the serial result as the reference for later comparisons.

We will use a small deterministic kernel.

In [ ]:
from time import perf_counter

def serial_sum_of_squares(n):
    total = 0
    for i in range(1, n + 1):
        total += i * i
    return total

N = 2_000_000
times = []

for rep in range(3):
    t0 = perf_counter()
    result_serial = serial_sum_of_squares(N)
    elapsed = perf_counter() - t0
    times.append(elapsed)
    print(f"Run {rep+1}: {elapsed:.4f} s")

T1 = min(times)
print(f"\nSerial baseline T1 = {T1:.4f} s")
print("Result:", result_serial)

### A better algorithm can beat more hardware

Remember the warning from the slides: slow code does not automatically mean HPC.

The same sum of squares has a closed-form formula:

```text
1^2 + 2^2 + ... + n^2 = n(n+1)(2n+1)/6
```

Predict what will matter more here:

- adding more processors to the loop; or
- replacing the loop with a better algorithm?

In [ ]:
def formula_sum_of_squares(n):
    return n * (n + 1) * (2*n + 1) // 6

t0 = perf_counter()
result_formula = formula_sum_of_squares(N)
formula_time = perf_counter() - t0

print("Same result:", result_formula == result_serial)
print(f"Loop baseline: {T1:.6f} s")
print(f"Closed form:   {formula_time:.9f} s")
if formula_time > 0:
    print(f"Observed improvement: about {T1/formula_time:,.0f}x")

### Explain

This is deliberately an extreme example, but the principle matters throughout HPC:

> **First improve the algorithm and establish a correct serial baseline. Then decide whether additional resources are justified.**

HPC is not a substitute for algorithmic thinking.

## 5 - Amdahl's Law: why more cores do not mean proportional speedup

Suppose **95%** of a program can be parallelized and **5%** is strictly serial.

Amdahl's Law is:

```text
S(N) = 1 / [s + (1-s)/N]
```

where:

- `s` = serial fraction;
- `N` = number of processors.

### Predict before running

Estimate the speedup for:

- 2 cores
- 4 cores
- 8 cores
- 16 cores
- infinitely many cores

With a 5% serial fraction, what is the maximum theoretical speedup?

In [ ]:
def amdahl_speedup(serial_fraction, processors):
    if processors == float("inf"):
        return 1.0 / serial_fraction
    return 1.0 / (serial_fraction + (1.0 - serial_fraction) / processors)

s = 0.05
processors = [1, 2, 4, 8, 16, 32, 64, float("inf")]

print(f"{'Processors':>12} {'Speedup':>12} {'Efficiency':>12}")
for p in processors:
    speedup = amdahl_speedup(s, p)
    if p == float("inf"):
        print(f"{'infinity':>12} {speedup:12.2f} {'-':>12}")
    else:
        efficiency = speedup / p
        print(f"{int(p):12d} {speedup:12.2f} {efficiency:12.2%}")

### Observe

Notice two things:

1. speedup continues to increase;
2. each doubling of resources gives **less additional benefit**.

With 5% serial work, infinite processors still cannot exceed **20x** theoretical speedup.

### Explain

More hardware cannot remove a serial bottleneck. Real systems also add overheads such as:

- synchronization;
- communication;
- load imbalance;
- data movement;
- runtime and scheduling overhead.

This is why "100 cores = 100x faster" is usually false.

## 6 - Performance is a whole-system outcome

Five resource categories can become the limiting factor:

1. **Compute** - CPU/GPU execution capability
2. **Memory** - capacity, bandwidth, latency, locality
3. **Network** - bandwidth, latency, topology
4. **Storage** - I/O rate, metadata, capacity
5. **Software** - compiler, libraries, runtime, scheduler

A faster component may make no difference if another part of the system is the bottleneck.

Let's model that idea with a simple pipeline.

In [ ]:
def pipeline_rate(**stages):
    limiting_stage = min(stages, key=stages.get)
    return stages[limiting_stage], limiting_stage

system = {
    "compute": 120.0,
    "memory": 80.0,
    "network": 25.0,
    "storage": 8.0,
    "software": 100.0,
}

rate, bottleneck = pipeline_rate(**system)

print("Illustrative stage capacities")
for name, value in system.items():
    print(f"  {name:<9}: {value:6.1f} units/s")

print(f"\nEnd-to-end sustained rate is limited to about {rate:.1f} units/s")
print("Bottleneck:", bottleneck)

### Predict

If we double only the compute capability from 120 to 240 units/s, what happens to the end-to-end rate?

In [ ]:
faster_compute = dict(system)
faster_compute["compute"] *= 2

old_rate, old_limit = pipeline_rate(**system)
new_rate, new_limit = pipeline_rate(**faster_compute)

print("Old end-to-end rate:", old_rate, "units/s - bottleneck:", old_limit)
print("New end-to-end rate:", new_rate, "units/s - bottleneck:", new_limit)

### Explain

Nothing useful happened because compute was not the limiting stage.

This is the point of the session's "balance" message:

> **Sustained application performance depends on the slowest relevant part of the end-to-end path.**

The hardware, software and algorithm must be matched to the workload.

## 7 - Bottleneck detective

For each workload, predict the **first likely bottleneck**. There can be more than one reasonable answer.

### Cases

**A. Dense matrix multiplication that fits in memory**  
**B. Scan 10 PB of scientific files**  
**C. Multi-node MPI program sending many tiny messages**  
**D. Very large in-memory graph traversal**  
**E. Program built with a poor compiler configuration that disables vectorization**

<details>
<summary><strong>Show explanation</strong></summary>

- **A:** usually **compute**, although memory movement still matters.
- **B:** usually **storage/I/O** and data movement.
- **C:** often **network latency/communication**.
- **D:** often **memory capacity/bandwidth/locality**.
- **E:** **software stack/compiler** can hide hardware capability.

Real applications can move from one bottleneck to another as they scale.

</details>

## 8 - First contact with the SciTech cluster

This section is **inspection only**. We will not submit a compute-heavy job in M1.S1.

### Goal

Collect evidence from the system itself:

1. What host are you on?
2. Who are you logged in as?
3. What CPU information can you see?
4. Which Slurm partitions and nodes are visible?
5. Is your Jupyter kernel already inside a Slurm allocation?
6. Can a login shell see Environment Modules?

> Keep credentials, passwords and tokens private. Never place them in notebooks or repositories.

In [ ]:
from pathlib import Path
import os
import subprocess
import shlex

def run_command(command):
    print("$", command)
    p = subprocess.run(
        command,
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )
    out = (p.stdout or "").strip()
    err = (p.stderr or "").strip()
    if out:
        print(out)
    if err:
        print("[stderr]")
        print(err)
    print("return code:", p.returncode)
    print()

print("Safe cluster inspection commands only. No heavy computation is launched.")

In [ ]:
run_command("hostname")
run_command("whoami")
run_command("pwd")
run_command("lscpu | head -n 12")

In [ ]:
if subprocess.run("command -v sinfo >/dev/null 2>&1", shell=True, executable="/bin/bash").returncode == 0:
    run_command("sinfo -a -N")
    run_command("squeue -u \"$USER\"")
else:
    print("Slurm commands are not available in this environment.")
    print("If you are using a local fallback, that is expected.")

In [ ]:
print("Selected Slurm environment variables visible to this Jupyter kernel:")
keys = [
    "SLURM_JOB_ID",
    "SLURM_JOB_NAME",
    "SLURM_JOB_PARTITION",
    "SLURM_NODELIST",
    "SLURM_CPUS_PER_TASK",
]
found = False
for key in keys:
    if key in os.environ:
        found = True
        print(f"{key}={os.environ[key]}")

if not found:
    print("No selected Slurm allocation variables were found.")

print("\nChecking whether a login shell exposes the module command:")
run_command("bash -lc 'type module >/dev/null 2>&1 && echo MODULE_COMMAND=AVAILABLE || echo MODULE_COMMAND=NOT_AVAILABLE'")

### Mini-challenge: evidence, not guesses

Write down a four-line answer from what your environment actually reported:

1. **Host:** ...
2. **CPU / cores visible:** ...
3. **Partitions / nodes visible:** ...
4. **One environment fact:** for example, whether `module` is available or whether Jupyter is already inside Slurm.

### Final question

Why should students avoid heavy computation in a shared login or interactive service?

<details>
<summary><strong>Show explanation</strong></summary>

Login and interactive services are shared entry points. Heavy work can interfere with other users and bypass the scheduler's resource controls.

In later sessions we will request resources explicitly through Slurm so computation runs where it belongs and the cluster can share expensive resources fairly.

</details>

## 9 - Exit ticket

Answer these in one sentence each.

### 1. What is HPC trying to optimize?

Your answer:

### 2. Name one likely application bottleneck.

Your answer:

### 3. Why keep a serial baseline?

Your answer:

### 4. If 5% of a program is strictly serial, why can infinite cores not produce infinite speedup?

Your answer:

<details>
<summary><strong>Show explanation</strong></summary>

- HPC is about meeting a performance goal for a workload that exceeds practical ordinary-system limits.
- Likely bottlenecks include compute, memory, network, storage and software.
- A serial baseline gives a correct, reproducible reference for any later speedup claim.
- Amdahl's Law says the serial fraction eventually dominates; 5% serial work limits theoretical speedup to 20x.

</details>

## What you should leave with

- HPC is defined by the **workload + performance goal**, not machine size alone.
- **Time-to-solution** and **throughput** are different objectives.
- Parallelism requires useful decomposable work.
- A trustworthy **serial baseline** comes before speedup.
- **Amdahl's Law** explains why the serial fraction eventually limits scaling.
- Compute, memory, network, storage and software must be balanced.
- Cluster work starts with observation and evidence, not guesses.

Next session we will go deeper into the architecture that makes these behaviors possible.